In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from functools import reduce  # Required in Python 3
import operator


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


# Data Processing

In [2]:
df = pd.read_csv('Car Dataset 1945-2020.csv', low_memory=False)

In [3]:
dfcolumns = df.columns
dfcolumns_indexes = {i : dfcolumns[i] for i in range(len(dfcolumns))}

In [4]:
# analyse every single column in this dataset.
for c in dfcolumns_indexes:
    col_name = dfcolumns_indexes[c]
    # check how many unique variables
    print(f"{col_name}: {df[col_name].nunique()}")

id_trim: 70823
Make: 255
Modle: 2706
Generation: 1742
Year_from: 97
Year_to: 94
Series: 1297
Trim: 10358
Body_type: 15
load_height_mm: 201
number_of_seats: 42
length_mm: 1677
width_mm: 585
height_mm: 840
wheelbase_mm: 834
front_track_mm: 469
rear_track_mm: 476
curb_weight_kg: 1979
wheel_size_r14: 635
ground_clearance_mm: 216
trailer_load_with_brakes_kg: 1369
payload_kg: 878
back_track_width_mm: 345
front_track_width_mm: 340
clearance_mm: 161
full_weight_kg: 1264
front_rear_axle_load_kg: 2133
max_trunk_capacity_l: 1415
cargo_compartment_length_width_height_mm: 244
cargo_volume_m3: 81
minimum_trunk_capacity_l: 876
maximum_torque_n_m: 748
injection_type: 12
overhead_camshaft: 1
cylinder_layout: 11
number_of_cylinders: 11
compression_ratio: 121
engine_type: 14
valves_per_cylinder: 6
boost_type: 9
cylinder_bore_mm: 55
stroke_cycle_mm: 59
engine_placement: 7
cylinder_bore_and_stroke_cycle_mm: 4
turnover_of_maximum_torque_rpm: 159
max_power_kw: 307
presence_of_intercooler: 1
capacity_cm3: 124

In [5]:
bad_data_indexes = [9,11,12,13,14,15,16,17,19,20,24,27,29,33,43,44,47,49,51,69,75,77]
bad_cols = df.columns[bad_data_indexes]

bad_cols

Index(['load_height_mm', 'length_mm', 'width_mm', 'height_mm', 'wheelbase_mm',
       'front_track_mm', 'rear_track_mm', 'curb_weight_kg',
       'ground_clearance_mm', 'trailer_load_with_brakes_kg', 'clearance_mm',
       'max_trunk_capacity_l', 'cargo_volume_m3', 'overhead_camshaft',
       'cylinder_bore_and_stroke_cycle_mm', 'turnover_of_maximum_torque_rpm',
       'capacity_cm3', 'engine_hp_rpm', 'bore_stroke_ratio', 'steering_type',
       'battery_capacity_KW_per_h', 'charging_time_h'],
      dtype='str')

In [6]:
for col_name in bad_cols:
    # check how many unique variables
    print(f"{col_name}: {df[col_name].nunique()}")

load_height_mm: 201
length_mm: 1677
width_mm: 585
height_mm: 840
wheelbase_mm: 834
front_track_mm: 469
rear_track_mm: 476
curb_weight_kg: 1979
ground_clearance_mm: 216
trailer_load_with_brakes_kg: 1369
clearance_mm: 161
max_trunk_capacity_l: 1415
cargo_volume_m3: 81
overhead_camshaft: 1
cylinder_bore_and_stroke_cycle_mm: 4
turnover_of_maximum_torque_rpm: 159
capacity_cm3: 1240
engine_hp_rpm: 147
bore_stroke_ratio: 2
steering_type: 1
battery_capacity_KW_per_h: 10
charging_time_h: 6


In [7]:
df = pd.read_csv('Car Dataset 1945-2020.csv', low_memory=False)
# Fix the naming for modle
df = df.rename(columns={"Modle": "model"})

bad_float_types = set(df.columns)

In [8]:
categorical_columns = [
    'Make',
    'drive_wheels',
    'Series',
    'boost_type',
    'back_suspension',
    'country_of_origin',
    'presence_of_intercooler',
    'Generation',
    'car_class',
    'Body_type',
    'rear_brakes',
    'rating_name',
    'emission_standards',
    'front_suspension',
    'steering_type',
    'injection_type',
    'transmission',
    'fuel_grade',
    'cylinder_layout',
    'front_brakes',
    'model',
    'engine_type',
    'overhead_camshaft'
    
]

cateforical_columns_check = ['engine_placement',
                             'Trim','number_of_seats',
                             'wheel_size_r14',
                             'front_rear_axle_load_kg',
                            'maximum_torque_n_m']
weird_columns = ['turnover_of_maximum_torque_rpm', 'range_km']

time_coumns = ['safety_assessment']


# drop all the categorical from bad_float_types
# We will do a separate analysis for this

for c in time_coumns:
    bad_float_types.discard(c)
for c in cateforical_columns_check:
    bad_float_types.discard(c)
for c in categorical_columns:
    bad_float_types.discard(c)
for c in weird_columns:
    bad_float_types.discard(c)
    
def is_numeric(value):
    try:
        float(value)
        return True
    except (TypeError, ValueError):
        return False

    
def cast_float_comma(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return 'nan'
    value = str(value)
    return value.replace(',','.') if ',' in value else value

def cast_float_space(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return 'nan'
    value = str(value)
    return value.replace(' ','') if ' ' in value else value

def cast_cylinder_bore_and_stroke_cycle_mm(value: str):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return 'nan'
    if not isinstance(value, str):
        value = str(value)
    if value == 'nan':
        return 'nan'
    if value:
        try:
            return 'x'.join([str(round((float(x.replace(',','.'))), 2)) for x in value.split('x')])
        except (TypeError, ValueError):
            return 'nan'
    return None


def cast_cargo_compartment_length_width_height_mm(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return 'nan'
    if not isinstance(value, str):
        value = str(value)
    if not value or value == 'nan':
        return 'nan'
    parts = value.split('x')
    if len(parts) != 3:
        return 'nan'
    try:
        return str(reduce(operator.mul, (float(item.replace(',', '.')) for item in parts), 1))
    except (TypeError, ValueError):
        return 'nan'
    
bad_comma_data = ['load_height_mm']
bad_space_data = ['clearance_mm']
bad_cylinder_bore_and_stroke_cycle_mm = ['cylinder_bore_and_stroke_cycle_mm'] 
bad_max_speed_km_per_h = ['max_speed_km_per_h'] 


# literally 1 row with DOHC, the rest are empty
bad_overhead_camshaft = ['overhead_camshaft']
bad_engine_hp_rpm = ['engine_hp_rpm']

bad_cargo_compartment_length_width_height_mm = ['cargo_compartment_length_width_height_mm']

for bad_col_name in bad_cargo_compartment_length_width_height_mm:
    df[bad_col_name] = df[bad_col_name].apply(cast_cargo_compartment_length_width_height_mm)

for bad_col_name in bad_engine_hp_rpm:
    df[bad_col_name] = df[bad_col_name].astype(str)
    df[bad_col_name] = df[bad_col_name].apply(lambda x: 'nan' if not isinstance(x, str) or '-' in x else x)

for bad_col_name in bad_max_speed_km_per_h:
    df[bad_col_name] = df[bad_col_name].astype(str)
    df[bad_col_name] = df[bad_col_name].apply(lambda x: 'nan' if not isinstance(x, str) or x == 'km/h' else x)

for bad_col_name in bad_cylinder_bore_and_stroke_cycle_mm:
    df[bad_col_name] = df[bad_col_name].astype(str)
    df[bad_col_name] = df[bad_col_name].apply(cast_cylinder_bore_and_stroke_cycle_mm)

for bad_col_name in bad_space_data:
    df[bad_col_name] = df[bad_col_name].astype(str)
    df[bad_col_name] = df[bad_col_name].apply(cast_float_space)

for bad_col_name in bad_float_types:
    df[bad_col_name] = df[bad_col_name].astype(str)
    df[bad_col_name] = df[bad_col_name].apply(cast_float_comma)

    
non_float_ignore = set(['cylinder_bore_and_stroke_cycle_mm'])


for ii, bad_col_name in enumerate(bad_float_types):
    if bad_col_name in non_float_ignore:
        continue
    
    df[bad_col_name] = df[bad_col_name].astype(str)
    mask =  df[bad_col_name].apply(is_numeric)
    non_numeric_values = df.loc[~mask,bad_col_name]
    if(len(non_numeric_values) >0) :
        print(bad_col_name, ii)
        print(non_numeric_values)
        
        break
    df[bad_col_name] = df[bad_col_name].astype(float)
print('run without errors!')



run without errors!


# Standardize Naming

In [9]:
# вывести названия всех текстовых колонок
text_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
print("Текстовые колонки:")
print(', '.join(text_cols))

Текстовые колонки:
Make, model, Generation, Series, Trim, Body_type, number_of_seats, wheel_size_r14, front_rear_axle_load_kg, maximum_torque_n_m, injection_type, overhead_camshaft, cylinder_layout, engine_type, boost_type, engine_placement, cylinder_bore_and_stroke_cycle_mm, turnover_of_maximum_torque_rpm, presence_of_intercooler, drive_wheels, transmission, range_km, emission_standards, fuel_grade, back_suspension, rear_brakes, front_brakes, front_suspension, steering_type, car_class, country_of_origin, safety_assessment, rating_name


In [10]:
df = df.rename(columns={"Modle": "model", 'cargo_compartment_length_width_height_mm': 'cargo_compartment_volume_mm3'})

# # Приводим к нижнему регистру (безопасно)
# cols_to_lower = [
#     'body_type', 'cylinder_layout', 'engine_type', 'boost_type',
#     'drive_wheels', 'transmission', 'emission_standards', 'fuel_grade',
#     'back_suspension', 'rear_brakes', 'front_brakes', 'front_suspension'
# ]

# standardize column names -> lowercase everything
df = df.rename(columns={c: c.lower() for c in df.columns })
categorical_columns = [x.lower() for x in categorical_columns]
cateforical_columns_check = [x.lower() for x in cateforical_columns_check]
weird_columns = [x.lower() for x in weird_columns]
time_coumns = [x.lower() for x in time_coumns]
df.head()

,id_trim,make,model,generation,year_from,year_to,series,trim,body_type,load_height_mm,...,front_suspension,steering_type,car_class,country_of_origin,number_of_doors,safety_assessment,rating_name,battery_capacity_kw_per_h,electric_range_km,charging_time_h
0,1.0,AC,ACE,1 generation,1993.0,2000.0,Cabriolet,3.5 MT,Cabriolet,NaN,...,Helical springs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,AC,ACE,1 generation,1993.0,2000.0,Cabriolet,4.6 MT,Cabriolet,NaN,...,Helical springs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,AC,ACE,1 generation,1993.0,2000.0,Cabriolet,4.9 AT,Cabriolet,NaN,...,Helical springs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,AC,ACE,1 generation,1993.0,2000.0,Roadster,2.9 AT,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,AC,ACE,1 generation,1993.0,2000.0,Roadster,2.9 MT,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# TODO Analyse these categorical variables

# categorical_columns
# cateforical_columns_check
# weird_columns
# time_coumns

In [12]:
# Lowercase everything (safe)

# Оставляем как есть (или проверяем вручную)
cols_to_keep_case = ['make', 'model', 'generation', 'series', 'trim']

# Build list of columns to normalize (only existing columns and not in cols_to_keep_case)
cols_to_normalize = [c for c in (categorical_columns + cateforical_columns_check + weird_columns) if c in df.columns and c not in cols_to_keep_case]

for cn in cols_to_normalize:
    # preserve existing NaNs, convert non-missing values to lowercase strings
    df[cn] = df[cn].where(df[cn].notna(), np.nan).astype(str).str.lower()
    # convert explicit 'nan' or empty strings back to NaN
    df.loc[df[cn].isin(['nan', '']), cn] = np.nan

# Report any columns that were missing and skipped
missing = [c for c in (categorical_columns + cateforical_columns_check + weird_columns) if c not in df.columns]
if missing:
    print('Missing columns skipped:', missing)

In [13]:
df.head()

,id_trim,make,model,generation,year_from,year_to,series,trim,body_type,load_height_mm,...,front_suspension,steering_type,car_class,country_of_origin,number_of_doors,safety_assessment,rating_name,battery_capacity_kw_per_h,electric_range_km,charging_time_h
0,1.0,AC,ACE,1 generation,1993.0,2000.0,Cabriolet,3.5 MT,cabriolet,NaN,...,helical springs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,AC,ACE,1 generation,1993.0,2000.0,Cabriolet,4.6 MT,cabriolet,NaN,...,helical springs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,AC,ACE,1 generation,1993.0,2000.0,Cabriolet,4.9 AT,cabriolet,NaN,...,helical springs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,AC,ACE,1 generation,1993.0,2000.0,Roadster,2.9 AT,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,AC,ACE,1 generation,1993.0,2000.0,Roadster,2.9 MT,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df.to_csv('cleaned.csv', index=False)